Justin Bec & Nicolas Beaujoin

# Variational Control for Guidance in Diffusion Models

In [ ]:
# Download files
!git clone https://github.com/DPS2022/diffusion-posterior-sampling.git
!cp -r diffusion-posterior-sampling/guided_diffusion guided_diffusion
!wget -nc -O ffhq256-1k-validation.zip 'https://www.dropbox.com/scl/fi/pppstbdsf0em6o0qscruc/ffhq256-1k-validation.zip?rlkey=xl7nwv2nxb6yvsirr3wad77hm'
!unzip -nq ffhq256-1k-validation.zip
!wget -nc -O ffhq_10m.pt 'https://www.dropbox.com/scl/fi/pq72vxzxcbygieq5z4gvf/ffhq_10m.pt?rlkey=5sxdj6r4o9f7b7bbp5fxg2f5r'

In [ ]:
import torch
import torchvision
import numpy as np

import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from guided_diffusion.unet import create_model
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

### Display functions (taken from the practical session)

In [ ]:
def tensor2im(x):
    x = 0.5+0.5*x # [-1,1]->[0,1]
    return x.detach().cpu().permute(2,3,1,0).squeeze()

def im2tensor(x):
    x = torch.tensor(x,device=device)
    x = 2*x-1 # [0,1]->[-1,1]
    return x.permute(2,0,1).unsqueeze(0)

def rgb2gray(u):
    return 0.2989 * u[:,:,0] + 0.5870 * u[:,:,1] + 0.1140 * u[:,:,2]

def str2(chars):
    return "{:.2f}".format(chars)

def psnr(uref,ut,M=2):
    rmse = np.sqrt(np.mean((np.array(uref.cpu())-np.array(ut.cpu()))**2))
    return 20*np.log10(M/rmse)

# viewimage
import tempfile
import IPython

def viewimage(im, normalize=True,vmin=-1,vmax=1,z=1,order=0,titre='',displayfilename=False):
    im = im.detach().cpu().permute(2,3,1,0).squeeze()
    imin= np.array(im).astype(np.float32)
    channel_axis = 2 if len(im.shape)>2 else None
    if z!=1:
        from skimage.transform import rescale
        imin = rescale(imin, z, order=order, channel_axis=channel_axis)
    if normalize:
        if vmin is None:
            vmin = imin.min()
        if vmax is None:
            vmax = imin.max()
        if np.abs(vmax-vmin)>1e-10:
            imin = (imin.clip(vmin,vmax)-vmin)/(vmax-vmin)
        else:
            imin = vmin
    else:
        imin=imin.clip(0,255)/255
    imin=(imin*255).astype(np.uint8)
    filename=tempfile.mktemp(titre+'.png')
    if displayfilename:
        print (filename)
    plt.imsave(filename, imin, cmap='gray')
    IPython.display.display(IPython.display.Image(filename))

idx = np.random.randint(1000)
print('Image', str(idx).zfill(5))

# Test display function:
img = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
print(img.min().item(),img.max().item())
viewimage(img)

### Load a pre-trained model on FFHQ

In [ ]:
# Load model
model_config = {'image_size': 256,
                'num_channels': 128,
                'num_res_blocks': 1,
                'channel_mult': '',
                'learn_sigma': True,
                'class_cond': False,
                'use_checkpoint': False,
                'attention_resolutions': 16,
                'num_heads': 4,
                'num_head_channels': 64,
                'num_heads_upsample': -1,
                'use_scale_shift_norm': True,
                'dropout': 0.0,
                'resblock_updown': True,
                'use_fp16': False,
                'use_new_attention_order': False,
                'model_path': 'ffhq_10m.pt'}
model = create_model(**model_config)
model = model.to(device)
model.eval();  # use in eval mode

In [ ]:
def get_eps_from_model(x, t):
    # the model outputs:
    # - an estimation of the noise eps (chanels 0 to 2)
    # - learnt variances for the posterior  (chanels 3 to 5)
    # (see Improved Denoising Diffusion Probabilistic Models
    # by Alex Nichol, Prafulla Dhariwal
    # for the parameterization)
    # We discard the second part of the output for this practice session.

    with torch.no_grad(): # avoid backprop wrt model parameters
        model_output = model(x, torch.tensor(t, device=device).unsqueeze(0))
    model_output = model_output[:,:3,:,:]
    return(model_output)


def get_eps_from_model_withgrad(x, t):
    # same as get_eps_from_model but kipping gradients
    model_output = model(x, torch.tensor(t, device=device).unsqueeze(0))
    model_output = model_output[:,:3,:,:]
    return(model_output)

### Reminder
First, we can take an image from FFHQ validation set and apply the forward model to it:
$$
\mathbf{x}_{t+1} = \sqrt{\alpha_t} \mathbf{x}_t + \sqrt{\beta_t} \mathbf{z}_t
$$

Recall that we can obtain a prediction of $x_0$ from $x_t$:
$$
\widehat{x_0}(\mathbf{x}_t) = \frac{1}{\sqrt{\bar{\alpha}_t}} \left(\mathbf{x}_t - \sqrt{\bar{\beta}_t} \; \varepsilon_t(\mathbf{x}_t, t) \right) ,
$$
which can also be written
$$
\widehat{x_0}(\mathbf{x}_t) = \frac{1}{\sqrt{\bar{\alpha}_t}} \mathbf{x}_t - \sqrt{\frac{1}{\bar{\alpha}_t}-1} \;  \varepsilon_t(\mathbf{x}_t, t).
$$

We can sample the backward model using the DDIM method:

$$ p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t) = \mathcal{N}(\mu_\theta(\mathbf{x}_t, t), \beta_t I_d) $$
where
$$
\mathbf{x}_{t-1} = \sqrt{\alpha_{t-1}} \left( \frac{\mathbf{x}_t - \sqrt{1 - \alpha_t}\epsilon_\theta(\mathbf{x}_t, t)}{\sqrt{\alpha_t}} \right) + \sqrt{1 - \alpha_{t-1}} \epsilon_\theta(\mathbf{x}_t, t)
$$

### Example of sampling with DDIM

In [ ]:
# Image 00462
idx = 463
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
imgshape = x0.shape

viewimage(x0)

In [ ]:
T_orig = 1000
beta_orig = np.linspace(0.0001, 0.02, T_orig, dtype=np.float64)
alpha_orig = 1.0 - beta_orig
alphabar_orig = np.cumprod(alpha_orig)

def make_ddim_schedule(num_steps=50, t_end=600):
    ddim_steps = np.linspace(0, t_end, num_steps + 1, dtype=int)
    ddim_steps = np.unique(ddim_steps)
    ddim_steps = ddim_steps[::-1].copy()
    alphabar_at = alphabar_orig[ddim_steps]
    alphabar_prev_at = np.append(alphabar_at[1:], 1.0)
    # IMPORTANT: alpha for DDIM steps = ratio of consecutive alphabars
    # NOT the per-step alpha from the original schedule
    alpha_ddim = alphabar_at / alphabar_prev_at
    return ddim_steps, alphabar_at, alphabar_prev_at, alpha_ddim

In [ ]:
def ddim_sample(imgshape, num_steps=100, t_end=999, eta=0.0):
    ddim_steps, ab, ab_prev, al = make_ddim_schedule(num_steps, t_end)
    x = torch.randn(imgshape, device=device)
    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],      device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i], device=device, dtype=torch.float32)
        eps_t = get_eps_from_model(x, t)
        xhat  = (x - torch.sqrt(1 - ab_t) * eps_t) / torch.sqrt(ab_t)
        xhat  = xhat.clamp(-1, 1)
        sigma_t = eta * torch.sqrt((1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t))
        dir_xt  = torch.sqrt(torch.clamp(1 - ab_p_t - sigma_t**2, min=0.0)) * eps_t
        noise   = sigma_t * torch.randn_like(x) if (sigma_t > 0 and i < len(ddim_steps)-1) else 0
        x = torch.sqrt(ab_p_t) * xhat + dir_xt + noise
        if i == 0 or i % (len(ddim_steps)//5) == 0 or i == len(ddim_steps)-1:
            print(f'Step {i}/{len(ddim_steps)} | t={t}')
            viewimage(torch.cat((x, xhat), dim=3))
    return x


# Test it:
# x_sample = ddim_sample(imgshape, num_steps=100, t_end=999, eta=0.0)

## 1. Deblurring on FFHQ with NDTM

Let's first try the algorithm on a deblurring task.

In [ ]:
import torch.nn.functional as F

def blur_operator(x, kernel):
    B, C, H, W = x.shape
    k = kernel.shape[0]
    pad = k // 2
    kernel_4d = kernel.view(1, 1, k, k).expand(C, 1, k, k)
    return F.conv2d(x, kernel_4d, padding=pad, groups=C)

def phi_deblurring(x0_hat, y_blurry, kernel):
    """Terminal cost: ||H(x0_hat) - y||^2"""
    Hx = blur_operator(x0_hat, kernel)
    return torch.sum((Hx - y_blurry) ** 2)

def make_gaussian_kernel(size=15, sigma=3.0, device='cpu'):
    coords = torch.arange(size, device=device) - size // 2
    g      = torch.exp(-coords**2 / (2 * sigma**2))
    kernel = torch.outer(g, g)
    return kernel / kernel.sum()


kernel = make_gaussian_kernel(size=15, sigma=3.0, device=device)


y_blurry = blur_operator(x0, kernel)
viewimage(torch.cat((x0, y_blurry), dim=3))

### NDTM algorithm

Here we can use a 'guided initialization', it means that the deblurring process does not start from pure noise but from a noisy version of the observation y_blurry. This trick is used in the official implementation but it is not mentioned in the paper.

In [ ]:
def guided_init_xT(y_observed, t_start, operator='blur', kernel=None,
                   mask=None, device='cuda'):
    """
    Initialize x_T from the observation instead of pure noise.
    This is what algo.init_xT=guided does in the official repo.

    For deblurring: x_T = sqrt(alphabar_T)*y_blurry + sqrt(1-alphabar_T)*noise
    For inpainting: x_T = sqrt(alphabar_T)*(M*y + (1-M)*noise_fill)
                         + sqrt(1-alphabar_T)*noise

    The idea: instead of starting from pure noise at t_end,
    start from a noisy version of the best rough estimate we have.
    The diffusion process then refines this rather than building
    from scratch.
    """
    ab_T = torch.tensor(alphabar_orig[t_start], device=device,
                        dtype=torch.float32)
    noise = torch.randn_like(y_observed)

    if operator == 'blur':
        # y_blurry is already a rough estimate of x0
        rough_estimate = y_observed
    elif operator == 'inpaint':
        # Fill masked regions with random noise
        rough_estimate = mask * y_observed + (1 - mask) * torch.randn_like(y_observed)
    else:
        rough_estimate = y_observed

    # Forward diffuse the rough estimate to timestep t_start
    x_T = torch.sqrt(ab_T) * rough_estimate + torch.sqrt(1 - ab_T) * noise
    return x_T

In [ ]:
def ndtm_deblurring_v4(
    y_blurry,
    blur_kernel,
    imgshape,
    num_steps=50,
    t_end=600,
    N=2,
    gamma=4.0,
    w_T=50.0,
    sigma_y=0.005,        # measurement noise level (from official repo)
    lr=0.01,
    eta=0.1,
    grad_clip=50.0,       # much more relaxed — let gradients work
    guided_init=True,     # use guided initialization
    device='cuda'
):
    ddim_steps, ab, ab_prev, al_ddim = make_ddim_schedule(num_steps, t_end)

    # --- Initialization ---
    if guided_init:
        x = guided_init_xT(y_blurry, ddim_steps[0],
                           operator='blur', device=device)
    else:
        x = torch.randn(imgshape, device=device)

    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],       device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i],  device=device, dtype=torch.float32)
        al_t   = torch.tensor(al_ddim[i],  device=device, dtype=torch.float32)

        # DDIM sigma
        sigma_ddim = eta * torch.sqrt(
            torch.clamp(
                (1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t),
                min=0.0
            )
        )

        # Cost weights from Proposition 3.1
        tau_t = (
            torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=1e-8))
            - torch.sqrt(torch.clamp(ab_p_t * (1 - al_t) / al_t, min=0.0))
        )
        kappa_t = gamma * torch.sqrt(ab_p_t / al_t)

        u = torch.zeros_like(x)

        with torch.no_grad():
            e_uncond = get_eps_from_model(x, t)

        # ---- Inner optimization ----
        for k in range(N):
            u = u.detach().requires_grad_(True)
            x_ctrl = x.detach() + gamma * u

            e_ctrl = get_eps_from_model_withgrad(x_ctrl, t)

            x0_hat = (
                (x_ctrl - torch.sqrt(1 - ab_t) * e_ctrl) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            C_score   = tau_t**2   * torch.sum((e_uncond - e_ctrl)**2)
            C_control = kappa_t**2 * torch.sum(u**2)

            # Terminal cost with sigma_y normalization
            residual = blur_operator(x0_hat, blur_kernel) - y_blurry
            C_term = (w_T / (2 * sigma_y**2)) * torch.mean(residual**2)
            # Using mean here for the terminal cost only, since
            # sigma_y^{-2} already provides the right scale.
            # Score and control use sum (they have their own weighting).

            C_total = C_score + C_control + C_term

            grad_u = torch.autograd.grad(C_total, u)[0]

            grad_norm = grad_u.norm()
            if grad_norm > grad_clip:
                grad_u = grad_u * (grad_clip / grad_norm)

            with torch.no_grad():
                # Linear LR decay
                lr_t = lr * (1.0 - i / len(ddim_steps))
                u = u - lr_t * grad_u

        # ---- DDIM step ----
        u_star = u.detach()
        x_guided = x.detach() + gamma * u_star

        with torch.no_grad():
            eps_g = get_eps_from_model(x_guided, t)
            x0_final = (
                (x_guided - torch.sqrt(1 - ab_t) * eps_g) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            dir_xt = (
                torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=0.0))
                * eps_g
            )
            noise = (
                sigma_ddim * torch.randn_like(x)
                if (sigma_ddim > 0 and i < len(ddim_steps) - 1)
                else 0
            )
            x = torch.sqrt(ab_p_t) * x0_final + dir_xt + noise

        if i == 0 or i % max(1, len(ddim_steps)//10) == 0 or i == len(ddim_steps)-1:
            u_norm = u_star.norm().item()
            print(
                f'Step {i}/{len(ddim_steps)} | t={t} | '
                f'C={C_total.item():.2f} '
                f'(score={C_score.item():.2f}, '
                f'ctrl={C_control.item():.2f}, '
                f'term={C_term.item():.2f}) | '
                f'||u||={u_norm:.4f}'
            )
            viewimage(torch.cat((x, x0_final, y_blurry), dim=3))

    return x

Test the algorithm. Let's just first try without the guided initialization to see how well it performs and we'll try to improve the performance afterwards.

In [ ]:
x0_restored = ndtm_deblurring_v4(
    y_blurry    = y_blurry,
    blur_kernel = kernel,
    imgshape    = imgshape,
    num_steps   = 50,       # official: 50
    t_end       = 600,      # official: 600
    N           = 2,        # official: M=2
    gamma       = 4.0,      # official: gamma_t=4.0
    w_T         = 50.0,     # official: w_terminal=50.0
    sigma_y     = 0.005,    # official: sigma_y=0.005
    lr          = 0.01,     # official: u_lr=0.01
    eta         = 0.0,      # official: eta=0.1
    grad_clip   = 50.0,     # relaxed to let gradients work
    guided_init = False,     # official: init_xT=guided
    device      = device,
)

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y_blurry, x0_restored],
    ['Ground truth', 'Blurry input', 'NDTM restored']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1,1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig("ndtm_result.png", dpi=150)
plt.show()
print(f"PSNR: {psnr(x0, x0_restored):.2f} dB")

Let's try to improve the NDTM algorithm by using the guided initialization.

In [ ]:
x0_restored = ndtm_deblurring_v4(
    y_blurry    = y_blurry,
    blur_kernel = kernel,
    imgshape    = imgshape,
    num_steps   = 50,       # official: 50
    t_end       = 600,      # official: 600
    N           = 2,        # official: M=2
    gamma       = 4.0,      # official: gamma_t=4.0
    w_T         = 50.0,     # official: w_terminal=50.0
    sigma_y     = 0.005,    # official: sigma_y=0.005
    lr          = 0.01,     # official: u_lr=0.01
    eta         = 0.0,      # official: eta=0.1
    grad_clip   = 50.0,     # relaxed to let gradients work
    guided_init = True,     # official: init_xT=guided
    device      = device,
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y_blurry, x0_restored],
    ['Ground truth', 'Blurry input', 'NDTM v4']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig("ndtm_v4_result.png", dpi=150)
plt.show()
print(f"PSNR: {psnr(x0, x0_restored):.2f} dB")

Now let's compare with the DPS algorith on the same task.

In [ ]:
def dps_deblurring_v4(
    y_blurry,
    blur_kernel,
    imgshape,
    num_steps=100,
    t_end=999,
    zeta=1.0,
    sigma_y=0.005,
    eta=0.0,
    guided_init=False,
    device='cuda'
):
    ddim_steps, ab, ab_prev, al_ddim = make_ddim_schedule(num_steps, t_end)

    if guided_init:
        x = guided_init_xT(y_blurry, ddim_steps[0],
                           operator='blur', device=device)
    else:
        x = torch.randn(imgshape, device=device)

    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],      device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i], device=device, dtype=torch.float32)

        x = x.detach().requires_grad_(True)

        eps_t = get_eps_from_model_withgrad(x, t)

        x0_hat = (
            (x - torch.sqrt(1 - ab_t) * eps_t) / torch.sqrt(ab_t)
        ).clamp(-1, 1)

        residual = blur_operator(x0_hat, blur_kernel) - y_blurry
        likelihood = torch.sum(residual ** 2) / (2 * sigma_y**2)

        grad_xt = torch.autograd.grad(likelihood, x)[0]

        with torch.no_grad():
            eps_clean = get_eps_from_model(x, t)
            x0_clean = (
                (x - torch.sqrt(1 - ab_t) * eps_clean) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            sigma_ddim = eta * torch.sqrt(
                torch.clamp(
                    (1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t),
                    min=0.0
                )
            )
            dir_xt = (
                torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=0.0))
                * eps_clean
            )
            noise = (
                sigma_ddim * torch.randn_like(x)
                if (sigma_ddim > 0 and i < len(ddim_steps) - 1)
                else 0
            )

            grad_norm = grad_xt.norm()
            guidance = zeta * grad_xt / grad_norm if grad_norm > 0 else 0

            x = torch.sqrt(ab_p_t) * x0_clean + dir_xt + noise - guidance

        if i == 0 or i % max(1, len(ddim_steps)//10) == 0 or i == len(ddim_steps)-1:
            print(
                f'Step {i}/{len(ddim_steps)} | t={t} | '
                f'loss={likelihood.item():.2f} | '
                f'||grad||={grad_norm.item():.4f}'
            )
            viewimage(torch.cat((x, x0_clean, y_blurry), dim=3))

    return x

First without guided initialization.

In [ ]:
print("="*60)
print("DEBLURRING — DPS v4")
print("="*60)
x0_dps_deblur = dps_deblurring_v4(
    y_blurry    = y_blurry,
    blur_kernel = kernel,
    imgshape    = imgshape,
    num_steps   = 100,
    t_end       = 999,
    zeta        = 1.0,
    sigma_y     = 0.005,
    eta         = 0.0,
    guided_init = False,
    device      = device,
)

Now let's try with guided initialization.

In [ ]:
x0_dps_deblur = dps_deblurring_v4(
    y_blurry    = y_blurry,
    blur_kernel = kernel,
    imgshape    = imgshape,
    num_steps   = 100,
    t_end       = 999,
    zeta        = 1.0,
    sigma_y     = 0.005,
    eta         = 0.0,
    guided_init = True,
    device      = device,
)

Compare the results of both methods.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y_blurry, x0_dps_deblur, x0_restored],
    ['Ground truth', 'Blurry', 'DPS v4', 'NDTM v4']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Deblurring', fontsize=14)
plt.tight_layout()
plt.savefig("deblurring_v4_comparison.png", dpi=150)
plt.show()

p_dps_d  = psnr(x0, x0_dps_deblur)
p_ndtm_d = psnr(x0, x0_restored)
print(f"Deblurring PSNR — DPS: {p_dps_d:.2f} dB | NDTM: {p_ndtm_d:.2f} dB")

Further evaluate on a few images.

In [ ]:
def evaluate_on_images(indices, kernel, method_fn, method_kwargs):
    """Run a method on a list of image indices, return avg PSNR."""
    psnrs = []
    for idx in indices:
        img = im2tensor(
            plt.imread(f'ffhq256-1k-validation/{str(idx).zfill(5)}.png')
        )
        y = blur_operator(img, kernel)
        restored = method_fn(
            y_blurry=y,
            blur_kernel=kernel,
            imgshape=img.shape,
            **method_kwargs
        )
        p = psnr(img, restored)
        psnrs.append(p)
        print(f'  Image {idx}: PSNR={p:.2f} dB')
    avg = np.mean(psnrs)
    print(f'  Average PSNR: {avg:.2f} dB')
    return psnrs, avg

In [ ]:
# Pick a few test images
test_indices = [42, 100, 200, 462, 750]

print("=== DPS ===")
psnrs_dps, avg_dps = evaluate_on_images(
    test_indices, kernel, dps_deblurring_v4,
    dict(num_steps=100, t_end=999, zeta=1.0, eta=0.0, device=device, guided_init=True, sigma_y=0.005)
)

print("\n=== NDTM ===")
psnrs_ndtm, avg_ndtm = evaluate_on_images(
    test_indices, kernel, ndtm_deblurring_v4,
    dict(num_steps=50, t_end=600, N=2, gamma=4.0, w_T=50.0,
         lr=0.01, eta=0.0, grad_clip=50.0, device=device, guided_init=True, sigma_y=0.005)
)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(test_indices))
w = 0.35
ax.bar(x_pos - w/2, psnrs_dps,  w, label=f'DPS (avg {avg_dps:.1f})')
ax.bar(x_pos + w/2, psnrs_ndtm, w, label=f'NDTM (avg {avg_ndtm:.1f})')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'#{i}' for i in test_indices])
ax.set_ylabel('PSNR (dB)')
ax.set_title('DPS vs NDTM — Gaussian Deblurring')
ax.legend()
plt.tight_layout()
plt.savefig("comparison_psnr.png", dpi=150)
plt.show()

### Let's try with a more complicated blur kernel (motion blur)

In [ ]:
# Clone le repo
!git clone https://github.com/czi-ai/oc-guidance.git

!pip install numpy pillow

import sys
sys.path.append('/content/oc-guidance')

from motionblur.motionblur import Kernel

from PIL import Image
from torchvision import transforms

kernel = Kernel(size=(61,61), intensity=0.8)
kernel.displayKernel()

# Extract as numpy array
k_np = kernel.kernelMatrix.astype(np.float32)

# Convert to torch tensor for use in operators
k_torch = torch.from_numpy(k_np).to(device)

import torch.nn.functional as F

idx = 361
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
imgshape = x0.shape

x_true = x0.clone()
print("original image")
viewimage(x_true)

sigma_noise = 2*10/255

y = blur_operator(x0,k_torch)
print("noisy measurement")
viewimage(y)
print(y.shape)
print(imgshape)

In [ ]:
x0_restored = ndtm_deblurring_v4(
    y_blurry    = y,
    blur_kernel = k_torch,
    imgshape    = imgshape,
    num_steps   = 50,       # official: 50
    t_end       = 600,      # official: 600
    N           = 2,        # official: M=2
    gamma       = 4.0,      # official: gamma_t=4.0
    w_T         = 50.0,     # official: w_terminal=50.0
    sigma_y     = 0.005,    # official: sigma_y=0.005
    lr          = 0.01,     # official: u_lr=0.01
    eta         = 0.0,      # official: eta=0.1
    grad_clip   = 50.0,     # relaxed to let gradients work
    guided_init = True,     # official: init_xT=guided
    device      = device,
)

In [ ]:
x0_dps_deblur = dps_deblurring_v4(
    y_blurry    = y,
    blur_kernel = k_torch,
    imgshape    = imgshape,
    num_steps   = 100,
    t_end       = 999,
    zeta        = 1.0,
    sigma_y     = 0.005,
    eta         = 0.0,
    guided_init = True,
    device      = device,
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y, x0_dps_deblur, x0_restored],
    ['Ground truth', 'Blurry (motion blur)', 'DPS', 'NDTM']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Deblurring', fontsize=14)
plt.tight_layout()
plt.savefig("deblurring_v4_comparison.png", dpi=150)
plt.show()

p_dps_d  = psnr(x0, x0_dps_deblur)
p_ndtm_d = psnr(x0, x0_restored)
print(f"Deblurring PSNR — DPS: {p_dps_d:.2f} dB | NDTM: {p_ndtm_d:.2f} dB")

## 2. Random inpainting

Now let's compare these algorithms on another inverse problem: random inpainting. We will do the same evaluation as for the previous task.

In [ ]:
def make_random_mask(shape, keep_ratio=0.1, device='cuda'):
    """
    Create a random binary mask.
    keep_ratio=0.1 means 10% of pixels are observed (90% masked).
    The mask is shared across channels (same pixels masked in R,G,B).
    """
    B, C, H, W = shape
    mask = (torch.rand(B, 1, H, W, device=device) < keep_ratio).float()
    mask = mask.expand(B, C, H, W)  # same spatial mask for all channels
    return mask


def inpainting_operator(x, mask):
    """Apply inpainting mask: y = M * x (element-wise)."""
    return mask * x

It is easy to adapt the previous algorithms (made for deblurring) to the inpainting task, we just have to change the terminal cost computation.

In [ ]:
def ndtm_inpainting_v4(
    y_masked,
    mask,
    imgshape,
    num_steps=50,
    t_end=600,
    N=2,
    gamma=4.0,
    w_T=50.0,
    sigma_y=0.005,
    lr=0.01,
    eta=0.1,
    grad_clip=50.0,
    guided_init=False,
    device='cuda'
):
    ddim_steps, ab, ab_prev, al_ddim = make_ddim_schedule(num_steps, t_end)

    if guided_init:
        x = guided_init_xT(y_masked, ddim_steps[0],
                           operator='inpaint', mask=mask, device=device)
    else:
        x = torch.randn(imgshape, device=device)

    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],       device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i],  device=device, dtype=torch.float32)
        al_t   = torch.tensor(al_ddim[i],  device=device, dtype=torch.float32)

        sigma_ddim = eta * torch.sqrt(
            torch.clamp(
                (1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t),
                min=0.0
            )
        )

        tau_t = (
            torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=1e-8))
            - torch.sqrt(torch.clamp(ab_p_t * (1 - al_t) / al_t, min=0.0))
        )
        kappa_t = gamma * torch.sqrt(ab_p_t / al_t)

        u = torch.zeros_like(x)

        with torch.no_grad():
            e_uncond = get_eps_from_model(x, t)

        for k in range(N):
            u = u.detach().requires_grad_(True)
            x_ctrl = x.detach() + gamma * u

            e_ctrl = get_eps_from_model_withgrad(x_ctrl, t)

            x0_hat = (
                (x_ctrl - torch.sqrt(1 - ab_t) * e_ctrl) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            C_score   = tau_t**2   * torch.sum((e_uncond - e_ctrl)**2)
            C_control = kappa_t**2 * torch.sum(u**2)

            residual = inpainting_operator(x0_hat, mask) - y_masked
            C_term = (w_T / (2 * sigma_y**2)) * torch.mean(residual**2)

            C_total = C_score + C_control + C_term

            grad_u = torch.autograd.grad(C_total, u)[0]

            grad_norm = grad_u.norm()
            if grad_norm > grad_clip:
                grad_u = grad_u * (grad_clip / grad_norm)

            with torch.no_grad():
                lr_t = lr * (1.0 - i / len(ddim_steps))
                u = u - lr_t * grad_u

        u_star = u.detach()
        x_guided = x.detach() + gamma * u_star

        with torch.no_grad():
            eps_g = get_eps_from_model(x_guided, t)
            x0_final = (
                (x_guided - torch.sqrt(1 - ab_t) * eps_g) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            dir_xt = (
                torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=0.0))
                * eps_g
            )
            noise = (
                sigma_ddim * torch.randn_like(x)
                if (sigma_ddim > 0 and i < len(ddim_steps) - 1)
                else 0
            )
            x = torch.sqrt(ab_p_t) * x0_final + dir_xt + noise

        if i == 0 or i % max(1, len(ddim_steps)//10) == 0 or i == len(ddim_steps)-1:
            u_norm = u_star.norm().item()
            print(
                f'Step {i}/{len(ddim_steps)} | t={t} | '
                f'C={C_total.item():.2f} '
                f'(score={C_score.item():.2f}, '
                f'ctrl={C_control.item():.2f}, '
                f'term={C_term.item():.2f}) | '
                f'||u||={u_norm:.4f}'
            )
            viewimage(torch.cat((x, x0_final, y_masked), dim=3))

    return x

In [ ]:
def dps_inpainting_v4(
    y_masked,
    mask,
    imgshape,
    num_steps=100,
    t_end=999,
    zeta=1.0,
    sigma_y=0.005,
    eta=0.0,
    guided_init=False,
    device='cuda'
):
    ddim_steps, ab, ab_prev, al_ddim = make_ddim_schedule(num_steps, t_end)

    if guided_init:
        x = guided_init_xT(y_masked, ddim_steps[0],
                           operator='inpaint', mask=mask, device=device)
    else:
        x = torch.randn(imgshape, device=device)

    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],      device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i], device=device, dtype=torch.float32)

        x = x.detach().requires_grad_(True)

        eps_t = get_eps_from_model_withgrad(x, t)

        x0_hat = (
            (x - torch.sqrt(1 - ab_t) * eps_t) / torch.sqrt(ab_t)
        ).clamp(-1, 1)

        residual = inpainting_operator(x0_hat, mask) - y_masked
        likelihood = torch.sum(residual ** 2) / (2 * sigma_y**2)

        grad_xt = torch.autograd.grad(likelihood, x)[0]

        with torch.no_grad():
            eps_clean = get_eps_from_model(x, t)
            x0_clean = (
                (x - torch.sqrt(1 - ab_t) * eps_clean) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            sigma_ddim = eta * torch.sqrt(
                torch.clamp(
                    (1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t),
                    min=0.0
                )
            )
            dir_xt = (
                torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=0.0))
                * eps_clean
            )
            noise = (
                sigma_ddim * torch.randn_like(x)
                if (sigma_ddim > 0 and i < len(ddim_steps) - 1)
                else 0
            )

            grad_norm = grad_xt.norm()
            guidance = zeta * grad_xt / grad_norm if grad_norm > 0 else 0

            x = torch.sqrt(ab_p_t) * x0_clean + dir_xt + noise - guidance

        if i == 0 or i % max(1, len(ddim_steps)//10) == 0 or i == len(ddim_steps)-1:
            print(
                f'Step {i}/{len(ddim_steps)} | t={t} | '
                f'loss={likelihood.item():.2f} | '
                f'||grad||={grad_norm.item():.4f}'
            )
            viewimage(torch.cat((x, x0_clean, y_masked), dim=3))

    return x

Let's pick another image.

In [ ]:
idx = 461
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
imgshape = x0.shape

In [ ]:
# Use the same image x0 and shape from earlier cells
torch.manual_seed(42)  # reproducible mask
mask = make_random_mask(imgshape, keep_ratio=0.1, device=device)
y_masked = inpainting_operator(x0, mask)

print("Observed pixels:", f"{mask[0,0].sum().item():.0f} / {mask[0,0].numel()}"
      f" ({100*mask[0,0].mean().item():.0f}%)")
viewimage(torch.cat((x0, y_masked), dim=3))

We will test directly the algorithms with guided initialization.

In [ ]:
# --- NDTM ---
print("\n" + "="*60)
print("NDTM Inpainting")
print("="*60)
x0_ndtm = ndtm_inpainting_v4(
    y_masked    = y_masked,
    mask        = mask,
    imgshape    = imgshape,
    num_steps   = 50,
    t_end       = 600,
    N           = 2,
    gamma       = 4.0,
    w_T         = 50.0,
    lr          = 0.01,
    eta         = 0.0,
    grad_clip   = 50.0,
    guided_init = True,
    device      = device,
)

In [ ]:
# --- DPS ---
print("\n" + "="*60)
print("DPS Inpainting")
print("="*60)
x0_dps = dps_inpainting_v4(
    y_masked    = y_masked,
    mask        = mask,
    imgshape    = imgshape,
    num_steps   = 100,
    t_end       = 999,
    zeta        = 1.0,
    eta         = 0.0,
    device      = device,
)


Compare

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y_masked, x0_dps, x0_ndtm],
    ['Ground truth', 'Masked (10% observed)', 'DPS', 'NDTM']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig("inpainting_comparison.png", dpi=150)
plt.show()

# Compute PSNR on full image
psnr_dps  = psnr(x0, x0_dps)
psnr_ndtm = psnr(x0, x0_ndtm)
print(f"PSNR  — DPS: {psnr_dps:.2f} dB | NDTM: {psnr_ndtm:.2f} dB")

# Also compute PSNR only on masked (unknown) pixels
mask_inv = 1.0 - mask
def psnr_masked(ref, est, m):
    diff = (ref - est) * m
    mse = (diff**2).sum() / m.sum()
    return 10 * torch.log10(torch.tensor(4.0) / mse).item()

psnr_dps_masked  = psnr_masked(x0, x0_dps, mask_inv)
psnr_ndtm_masked = psnr_masked(x0, x0_ndtm, mask_inv)
print(f"PSNR (masked pixels only) — DPS: {psnr_dps_masked:.2f} dB"
      f" | NDTM: {psnr_ndtm_masked:.2f} dB")

In [ ]:
test_indices = [42, 100, 200, 462, 750]
results = {'dps': [], 'ndtm': []}

for idx in test_indices:
    img = im2tensor(
        plt.imread(f'ffhq256-1k-validation/{str(idx).zfill(5)}.png')
    )
    torch.manual_seed(idx)  # reproducible per-image mask
    m = make_random_mask(img.shape, keep_ratio=0.1, device=device)
    y = inpainting_operator(img, m)

    print(f"\n--- Image {idx} ---")

    res_dps = dps_inpainting_v4(
        y, m, img.shape, num_steps=100, t_end=999,
        zeta=1.0, device=device, eta=0.0, guided_init=True
    )
    p_dps = psnr(img, res_dps)
    results['dps'].append(p_dps)
    print(f"  DPS:  {p_dps:.2f} dB")

    res_ndtm = ndtm_inpainting_v4(
        y, m, img.shape, num_steps=50, t_end=600,
        N=2, gamma=4.0, w_T=50.0, lr=0.01, eta=0.0,
        grad_clip=50.0, device=device, guided_init=True
    )
    p_ndtm = psnr(img, res_ndtm)
    results['ndtm'].append(p_ndtm)
    print(f"  NDTM: {p_ndtm:.2f} dB")

In [ ]:
# Summary
print("\n" + "="*60)
print(f"Average PSNR — DPS: {np.mean(results['dps']):.2f} dB"
      f" | NDTM: {np.mean(results['ndtm']):.2f} dB")

fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(test_indices))
w = 0.35
ax.bar(x_pos - w/2, results['dps'],  w, label=f"DPS (avg {np.mean(results['dps']):.1f})")
ax.bar(x_pos + w/2, results['ndtm'], w, label=f"NDTM (avg {np.mean(results['ndtm']):.1f})")
ax.set_xticks(x_pos)
ax.set_xticklabels([f'#{i}' for i in test_indices])
ax.set_ylabel('PSNR (dB)')
ax.set_title('DPS vs NDTM — Random Inpainting (90% masked)')
ax.legend()
plt.tight_layout()
plt.savefig("inpainting_psnr_comparison.png", dpi=150)
plt.show()

## 3. Large Area Inpainting

Here we can use the functions from the previous part and just modify the mask.

In [ ]:
idx = 365
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
imgshape = x0.shape
viewimage(x0)

In [ ]:
h = imgshape[2]
w = imgshape[3]
hcrop, wcrop = h//2, w//2
corner_top, corner_left = h//4, int(0.45*w)
mask = torch.ones(imgshape, device=device)
mask[:,:,corner_top:corner_top+hcrop,corner_left:corner_left+wcrop] = 0

sigma_noise = 2*10/255

y = inpainting_operator(x0.clone(), mask) + sigma_noise * mask * torch.randn_like(x0)
viewimage(torch.cat((x0, y), dim=3))

In [ ]:
# --- NDTM ---
print("\n" + "="*60)
print("NDTM Large Area Inpainting")
print("="*60)
x0_ndtm = ndtm_inpainting_v4(
    y_masked    = y,
    mask        = mask,
    imgshape    = imgshape,
    num_steps   = 50,
    t_end       = 600,
    N           = 2,
    gamma       = 4.0,
    w_T         = 50.0,
    lr          = 0.01,
    eta         = 0.0,
    grad_clip   = 50.0,
    guided_init = True,
    device      = device,
)

In [ ]:
# --- DPS ---
print("\n" + "="*60)
print("DPS Large Area Inpainting")
print("="*60)
x0_dps = dps_inpainting_v4(
    y_masked    = y,
    mask        = mask,
    imgshape    = imgshape,
    num_steps   = 100,
    t_end       = 999,
    zeta        = 1.0,
    eta         = 0.0,
    guided_init = True,
    device      = device,
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y, x0_dps, x0_ndtm],
    ['Ground truth', 'Area Inpainting', 'DPS', 'NDTM']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig("dps_vs_ndtm.png", dpi=150)
plt.show()

psnr_dps  = psnr(x0, x0_dps)
psnr_ndtm = psnr(x0, x0_ndtm)
print(f"PSNR  — DPS: {psnr_dps:.2f} dB | NDTM: {psnr_ndtm:.2f} dB")

## 4. Non-Linear Deblurring

In [ ]:
def nonlinear_blur_operator(x, kernel):
    """Blur + bicubic downsample + bicubic upsample = non-linear."""
    B, C, H, W = x.shape
    k = kernel.shape[0]
    pad = k // 2
    kernel_4d = kernel.view(1, 1, k, k).expand(C, 1, k, k)

    # Step 1: linear convolution
    blurred = F.conv2d(x, kernel_4d, padding=pad, groups=C)

    # Step 2: bicubic downsample (NON-LINEAR — this is what breaks linearity)
    down = F.interpolate(blurred, scale_factor=0.125, mode='bicubic',
                         align_corners=False)

    # Step 3: bicubic upsample back to original size (also non-linear)
    up = F.interpolate(down, size=(H, W), mode='bicubic',
                       align_corners=False)
    return up

In [ ]:
kernel_nl = make_gaussian_kernel(size=61, sigma=3.0, device=device)

y_nonlinear = nonlinear_blur_operator(x0, kernel_nl)

viewimage(torch.cat((x0, y_nonlinear), dim=3))

In [ ]:
def ndtm_deblurring_non_linear(
    y_blurry,
    blur_kernel,
    imgshape,
    num_steps=50,
    t_end=600,
    N=2,
    gamma=4.0,
    w_T=50.0,
    sigma_y=0.005,        # measurement noise level (from official repo)
    lr=0.01,
    eta=0.1,
    grad_clip=50.0,       # much more relaxed — let gradients work
    guided_init=True,     # use guided initialization
    device='cuda'
):
    ddim_steps, ab, ab_prev, al_ddim = make_ddim_schedule(num_steps, t_end)

    # --- Initialization ---
    if guided_init:
        x = guided_init_xT(y_blurry, ddim_steps[0],
                           operator='blur', device=device)
    else:
        x = torch.randn(imgshape, device=device)

    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],       device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i],  device=device, dtype=torch.float32)
        al_t   = torch.tensor(al_ddim[i],  device=device, dtype=torch.float32)

        # DDIM sigma
        sigma_ddim = eta * torch.sqrt(
            torch.clamp(
                (1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t),
                min=0.0
            )
        )

        # Cost weights from Proposition 3.1
        tau_t = (
            torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=1e-8))
            - torch.sqrt(torch.clamp(ab_p_t * (1 - al_t) / al_t, min=0.0))
        )
        kappa_t = gamma * torch.sqrt(ab_p_t / al_t)

        u = torch.zeros_like(x)

        with torch.no_grad():
            e_uncond = get_eps_from_model(x, t)

        # ---- Inner optimization ----
        for k in range(N):
            u = u.detach().requires_grad_(True)
            x_ctrl = x.detach() + gamma * u

            e_ctrl = get_eps_from_model_withgrad(x_ctrl, t)

            x0_hat = (
                (x_ctrl - torch.sqrt(1 - ab_t) * e_ctrl) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            C_score   = tau_t**2   * torch.sum((e_uncond - e_ctrl)**2)
            C_control = kappa_t**2 * torch.sum(u**2)

            # Terminal cost with sigma_y normalization
            residual = nonlinear_blur_operator(x0_hat, blur_kernel) - y_blurry
            C_term = (w_T / (2 * sigma_y**2)) * torch.mean(residual**2)
            # Using mean here for the terminal cost only, since
            # sigma_y^{-2} already provides the right scale.
            # Score and control use sum (they have their own weighting).

            C_total = C_score + C_control + C_term

            grad_u = torch.autograd.grad(C_total, u)[0]

            grad_norm = grad_u.norm()
            if grad_norm > grad_clip:
                grad_u = grad_u * (grad_clip / grad_norm)

            with torch.no_grad():
                # Linear LR decay
                lr_t = lr * (1.0 - i / len(ddim_steps))
                u = u - lr_t * grad_u

        # ---- DDIM step ----
        u_star = u.detach()
        x_guided = x.detach() + gamma * u_star

        with torch.no_grad():
            eps_g = get_eps_from_model(x_guided, t)
            x0_final = (
                (x_guided - torch.sqrt(1 - ab_t) * eps_g) / torch.sqrt(ab_t)
            ).clamp(-1, 1)

            dir_xt = (
                torch.sqrt(torch.clamp(1 - ab_p_t - sigma_ddim**2, min=0.0))
                * eps_g
            )
            noise = (
                sigma_ddim * torch.randn_like(x)
                if (sigma_ddim > 0 and i < len(ddim_steps) - 1)
                else 0
            )
            x = torch.sqrt(ab_p_t) * x0_final + dir_xt + noise

        if i == 0 or i % max(1, len(ddim_steps)//10) == 0 or i == len(ddim_steps)-1:
            u_norm = u_star.norm().item()
            print(
                f'Step {i}/{len(ddim_steps)} | t={t} | '
                f'C={C_total.item():.2f} '
                f'(score={C_score.item():.2f}, '
                f'ctrl={C_control.item():.2f}, '
                f'term={C_term.item():.2f}) | '
                f'||u||={u_norm:.4f}'
            )
            viewimage(torch.cat((x, x0_final, y_blurry), dim=3))

    return x

In [ ]:
x0_restored = ndtm_deblurring_non_linear(
    y_blurry    = y_nonlinear,
    blur_kernel = kernel_nl,
    imgshape    = imgshape,
    num_steps   = 50,
    t_end       = 500,
    N           = 2,
    gamma       = 4.0,
    w_T         = 75.,     # back to paper's value
    lr          = 0.01,
    eta         = 0.0,
    grad_clip   = 10.0,
    guided_init = True,    # relaxed from 1.0
    device      = device,
)

In [ ]:
def dps_deblurring_non_linear(
    y_blurry,
    blur_kernel,
    imgshape,
    num_steps=100,        # DPS typically needs more steps than NDTM
    t_end=999,            # start from full noise (standard for DPS)
    zeta=1.0,             # guidance step size (main knob to tune)
    eta=0.0,              # DDIM stochasticity (DPS paper uses eta=0)
    clamp_x0=True,
    device='cuda'
):
    ddim_steps, ab, ab_prev, al_ddim = make_ddim_schedule(num_steps, t_end)

    x = torch.randn(imgshape, device=device)

    for i, t in enumerate(ddim_steps):
        ab_t   = torch.tensor(ab[i],      device=device, dtype=torch.float32)
        ab_p_t = torch.tensor(ab_prev[i], device=device, dtype=torch.float32)

        # --- Need gradients w.r.t. x_t for the guidance term ---
        x = x.detach().requires_grad_(True)

        # Forward pass through the model (with grad)
        eps_t = get_eps_from_model_withgrad(x, t)

        # Tweedie estimate of x0
        x0_hat = ((x - torch.sqrt(1 - ab_t) * eps_t) / torch.sqrt(ab_t))
        if clamp_x0:
            x0_hat = x0_hat.clamp(-1, 1)

        # Measurement likelihood: ||A(x0_hat) - y||^2
        residual = nonlinear_blur_operator(x0_hat, blur_kernel) - y_blurry
        likelihood = torch.sum(residual ** 2)

        # Gradient of likelihood w.r.t. x_t
        grad_xt = torch.autograd.grad(likelihood, x)[0]

        # --- Standard DDIM step (no grad needed from here) ---
        with torch.no_grad():
            # Recompute eps without grad for clean DDIM step
            eps_t_clean = get_eps_from_model(x, t)
            x0_hat_clean = (
                (x - torch.sqrt(1 - ab_t) * eps_t_clean) / torch.sqrt(ab_t)
            )
            if clamp_x0:
                x0_hat_clean = x0_hat_clean.clamp(-1, 1)

            # DDIM update
            sigma_t = eta * torch.sqrt(
                torch.clamp(
                    (1 - ab_p_t) / (1 - ab_t) * (1 - ab_t / ab_p_t),
                    min=0.0
                )
            )
            dir_xt = (
                torch.sqrt(torch.clamp(1 - ab_p_t - sigma_t**2, min=0.0))
                * eps_t_clean
            )
            noise = (
                sigma_t * torch.randn_like(x)
                if (sigma_t > 0 and i < len(ddim_steps) - 1)
                else 0
            )

            # DPS guidance: subtract scaled gradient
            # Normalize by the norm to stabilize (DPS trick)
            grad_norm = grad_xt.norm()
            if grad_norm > 0:
                guidance = zeta * grad_xt / grad_norm
            else:
                guidance = 0

            x_prev = torch.sqrt(ab_p_t) * x0_hat_clean + dir_xt + noise
            x = x_prev - guidance

        if i == 0 or i % max(1, len(ddim_steps) // 10) == 0 or i == len(ddim_steps) - 1:
            print(
                f'Step {i}/{len(ddim_steps)} | t={t} | '
                f'||A(x0)-y||^2={likelihood.item():.2f} | '
                f'||grad||={grad_norm.item():.4f}'
            )
            viewimage(torch.cat((x, x0_hat_clean, y_blurry), dim=3))

    return x

In [ ]:
x0_dps = dps_deblurring_non_linear(
    y_blurry    = y_nonlinear,
    blur_kernel = kernel_nl,
    imgshape    = imgshape,
    num_steps   = 100,     # DPS needs more steps (100-1000)
    t_end       = 999,     # full noise schedule
    zeta        = 1.0,     # guidance weight — tune this first
    eta         = 0.0,     # deterministic DDIM
    device      = device,
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tensor, title in zip(
    axes,
    [x0, y_nonlinear, x0_dps, x0_restored],
    ['Ground truth', 'Blurry (non linear)', 'DPS', 'NDTM']
):
    disp = (tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) / 2
    ax.imshow(disp.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig("dps_vs_ndtm.png", dpi=150)
plt.show()

psnr_dps  = psnr(x0, x0_dps)
psnr_ndtm = psnr(x0, x0_restored)
print(f"PSNR  — DPS: {psnr_dps:.2f} dB | NDTM: {psnr_ndtm:.2f} dB")